# 06 SenseVoice 数据增强与数据量消融实验 — 模型评估

评估 8 个微调模型在验证集上的 CER：
1. **SV-base** — SenseVoice 基座模型（无微调）
2. **SV-25%** — 25% 数据量微调
3. **SV-50%** — 50% 数据量微调
4. **SV-75%** — 75% 数据量微调
5. **SV-noiseA** — 高斯合成噪音增强（100% 数据）
6. **SV-noiseB** — ESC-50 真实噪音增强（100% 数据）
7. **SV-mixedA** — 原始数据 + 高斯合成噪音（1:1 混合）
8. **SV-mixedB** — 原始数据 + ESC-50 真实噪音（1:1 混合）

## 1. 环境检查

In [1]:
import warnings
warnings.filterwarnings('ignore')

import os, re, json, time, gc
import torch
import numpy as np

device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'PyTorch: {torch.__version__}')
print(f'Device: {device}')

import funasr
print(f'FunASR: {funasr.__version__}')

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
print(f'matplotlib: OK')

print('--- 路径检查 ---')
REPORT_DIR = os.path.expanduser('~/Projects/Agent/local')
MODEL_DIR = os.path.expanduser('~/Projects/Agent/model')
VAL_JSONL = os.path.expanduser('~/Desktop/hengdong_asr_trainset/manifests/val.jsonl')
print(f'REPORT_DIR: {REPORT_DIR}')
print(f'MODEL_DIR: {MODEL_DIR}')
print(f'VAL_JSONL exists: {os.path.exists(VAL_JSONL)}')

PyTorch: 2.11.0
Device: mps
FunASR: 1.3.1
matplotlib: OK
--- 路径检查 ---
REPORT_DIR: /Users/fanhua/Projects/Agent/local
MODEL_DIR: /Users/fanhua/Projects/Agent/model
VAL_JSONL exists: True


## 2. 工具函数

In [2]:
def free_memory():
    gc.collect()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()
    elif torch.cuda.is_available():
        torch.cuda.empty_cache()

# 打印当前内存状态
import psutil
def mem_info():
    mem = psutil.virtual_memory()
    print(f'  内存: {mem.used/1024**3:.1f}GB / {mem.total/1024**3:.1f}GB (使用 {mem.percent}%)')

def clean_sensevoice_text(text):
    """去除 SenseVoice 特殊标记"""
    return re.sub(r'<\|[^|]*\|>', '', text).strip()

_PUNCT = re.compile(
    r'[\s\.,!?;:\-()\[\]{}\u3000\uff0c\u3002\uff01\uff1f\u3001\uff1b\uff1a'
    r'\u201c\u201d\u2018\u2019\uff08\uff09\u3010\u3011\u300a\u300b\u2026\u2014]'
)
def norm_punct(text):
    """去除标点符号（用于 CER 计算）"""
    return _PUNCT.sub('', text)

def levenshtein(s1, s2):
    """计算编辑距离"""
    if len(s1) < len(s2):
        s1, s2 = s2, s1
    if len(s2) == 0:
        return len(s1)
    prev = list(range(len(s2) + 1))
    for c1 in s1:
        curr = [prev[0] + 1]
        for j, c2 in enumerate(s2):
            curr.append(min(prev[j + 1] + 1, curr[j] + 1, prev[j] + (c1 != c2)))
        prev = curr
    return prev[-1]

print('工具函数就绪')

工具函数就绪


## 3. 加载验证集

In [3]:
DATA_ROOT = os.path.expanduser('~/Desktop/hengdong_asr_trainset')
VAL_JSONL = os.path.join(DATA_ROOT, 'manifests/val.jsonl')

samples = []
with open(VAL_JSONL, 'r', encoding='utf-8') as f:
    for line in f:
        s = json.loads(line)
        audio = s.get('audio_filepath') or s.get('source')
        if audio and os.path.exists(audio):
            samples.append(s)

print(f'验证集: {len(samples)} 条有效样本')

验证集: 2261 条有效样本


## 4. 评估函数

In [4]:
from funasr import AutoModel

def eval_model(model, samples, label):
    """评估单个模型，返回 CER 统计（CER 计算前先去除标点）"""
    results = []
    total_cer, total_chars, exact = 0.0, 0, 0
    start = time.time()
    with torch.inference_mode():
        for i, s in enumerate(samples):
            audio = s.get('audio_filepath') or s.get('source')
            expected_raw = s.get('text') or s.get('target')
            if not audio or not os.path.exists(audio):
                continue
            try:
                res = model.generate(input=audio, language='auto', use_itn=True)
                pred_raw = clean_sensevoice_text(res[0]['text']) if res else ''
            except:
                pred_raw = ''
            # 标点归一化后计算 CER
            expected = norm_punct(expected_raw)
            pred = norm_punct(pred_raw)
            dist = levenshtein(expected, pred)
            ref_len = max(len(expected), 1)
            cer = dist / ref_len
            total_cer += dist
            total_chars += ref_len
            if expected == pred:
                exact += 1
            results.append({'id': i, 'expected': expected, 'predicted': pred, 'cer': cer})
            if (i + 1) % 500 == 0:
                print(f'    {label}: {i+1}/{len(samples)} | CER={total_cer/total_chars:.2%}')
    cer = total_cer / total_chars if total_chars > 0 else 0
    elapsed = time.time() - start
    exact_rate = exact / len(results) if results else 0
    print(f'  {label}: CER={cer:.2%} exact={exact}/{len(results)} time={elapsed:.1f}s')
    return {
        'name': label, 'cer': cer, 'exact': exact,
        'exact_rate': exact_rate, 'total': len(results),
        'time': elapsed, 'results': results
    }

print('评估函数就绪')

评估函数就绪


## 5. 模型配置

所有 checkpoint 均存储于 OSS `corpus/output/` 下，评估时直接通过 `model_only=True` + `init_param` 加载。

In [5]:
MODEL_PATHS = {
    # 100% 原数据微调（最先跑）
    'SV-ft':       os.path.join(MODEL_DIR, 'sensevoice_lora', 'model.pt.best'),
    # 100% 数据量增强实验
    'SV-noiseA':   os.path.join(MODEL_DIR, 'sv_noise_a', 'model.pt.best'),
    'SV-noiseB':   os.path.join(MODEL_DIR, 'sv_noise_b', 'model.pt.best'),
    'SV-mixedA':   os.path.join(MODEL_DIR, 'sv_mixed_a', 'model.pt.best'),
    'SV-mixedB':   os.path.join(MODEL_DIR, 'sv_mixed_b', 'model.pt.best'),
    # 数据量消融实验
    'SV-25%':      os.path.join(MODEL_DIR, 'sv_25pct', 'model.pt.best'),
    'SV-50%':      os.path.join(MODEL_DIR, 'sv_50pct', 'model.pt.best'),
    'SV-75%':      os.path.join(MODEL_DIR, 'sv_75pct', 'model.pt.best'),
    # 基座模型（最后跑）
    'SV-base':     None,
}

# 检查本地模型是否存在
print('本地模型检查:')
for name, path in MODEL_PATHS.items():
    if path is None:
        print(f'  {name}: HuggingFace 基座模型')
    else:
        exists = os.path.exists(path)
        size = os.path.getsize(path) / 1024 / 1024 if exists else 0
        status = 'OK' if exists else 'MISSING'
        print(f'  {name}: {status} ({size:.0f}MB) - {path}')

本地模型检查:
  SV-ft: OK (2679MB) - /Users/fanhua/Projects/Agent/model/sensevoice_lora/model.pt.best
  SV-noiseA: OK (2679MB) - /Users/fanhua/Projects/Agent/model/sv_noise_a/model.pt.best
  SV-noiseB: OK (2679MB) - /Users/fanhua/Projects/Agent/model/sv_noise_b/model.pt.best
  SV-mixedA: OK (2679MB) - /Users/fanhua/Projects/Agent/model/sv_mixed_a/model.pt.best
  SV-mixedB: OK (2679MB) - /Users/fanhua/Projects/Agent/model/sv_mixed_b/model.pt.best
  SV-25%: OK (2679MB) - /Users/fanhua/Projects/Agent/model/sv_25pct/model.pt.best
  SV-50%: OK (2679MB) - /Users/fanhua/Projects/Agent/model/sv_50pct/model.pt.best
  SV-75%: OK (2679MB) - /Users/fanhua/Projects/Agent/model/sv_75pct/model.pt.best
  SV-base: HuggingFace 基座模型


## 6. 串行评估（依次加载每个模型）

In [6]:
import time
from funasr import AutoModel

all_results = []

for name, model_path in MODEL_PATHS.items():
    print(f'\n{"="*60}\n评估 [{name}]\n{"="*60}')
    mem_info()
    
    if name == 'SV-base':
        # 基座模型，直接从 HuggingFace 加载
        model = AutoModel(model='iic/SenseVoiceSmall', disable_update=True, device=device)
        
    elif name == 'SV-ft':
        # 早期微调模型：LoRA checkpoint，需用 lora_only=True + torch.load
        if not os.path.exists(model_path):
            print(f'  跳过: 模型文件不存在 {model_path}')
            continue
        model = AutoModel(model='iic/SenseVoiceSmall', lora_only=True, disable_update=True, device=device)
        ckpt = torch.load(model_path, map_location='cpu', weights_only=False)
        model.model.load_state_dict(ckpt['state_dict'], strict=False)
        del ckpt
        print(f'  微调权重已载入: {model_path}')
        
    else:
        # 消融实验模型（sv_25pct/sv_50pct 等）：完整 checkpoint（base + lora 合并）
        if not os.path.exists(model_path):
            print(f'  跳过: 模型文件不存在 {model_path}')
            continue
        model = AutoModel(
            model='iic/SenseVoiceSmall',
            disable_update=True,
            device=device,
            init_param=model_path,
        )
    
    result = eval_model(model, samples, name)
    all_results.append(result)
    del model
    free_memory()
    time.sleep(2)
    mem_info()
    print(f'  完成: CER={result["cer"]:.2%}')


评估 [SV-ft]
  内存: 6.0GB / 16.0GB (使用 53.3%)
funasr version: 1.3.1.


  微调权重已载入: /Users/fanhua/Projects/Agent/model/sensevoice_lora/model.pt.best


rtf_avg: 0.123: 100%|██████████| 1/1 [00:00<00:00, 11.20it/s]                                                                                          


    SV-ft: 500/2261 | CER=21.42%


rtf_avg: 0.295: 100%|██████████| 1/1 [00:00<00:00,  8.01it/s]                                                                                          


...

    SV-base: 1500/2261 | CER=68.37%


rtf_avg: 0.033: 100%|██████████| 1/1 [00:00<00:00,  7.99it/s]                                                                                          


    SV-base: 2000/2261 | CER=67.51%


rtf_avg: 0.107: 100%|██████████| 1/1 [00:00<00:00,  9.06it/s]                                                                                          


  SV-base: CER=63.71% exact=123/2261 time=344.3s
  内存: 6.9GB / 16.0GB (使用 59.1%)
  完成: CER=63.71%


## 7. 保存结果

In [7]:
result_path = os.path.join(REPORT_DIR, 'sv_ablation_eval.json')
with open(result_path, 'w', encoding='utf-8') as f:
    json.dump({
        'results': [
            {
                'name': r['name'],
                'cer': round(r['cer'], 4),
                'exact_rate': round(r['exact_rate'], 4),
                'exact': r['exact'],
                'total': r['total'],
                'time_sec': round(r['time'], 1),
            } for r in all_results
        ]
    }, f, ensure_ascii=False, indent=2)
print(f'结果已保存: {result_path}')

结果已保存: /Users/fanhua/Projects/Agent/local/sv_ablation_eval.json
